# 03 · Understand the embeddings, then export the map

This is the actual point of the project: don't just treat the 1024-dim Clay vectors as a black box, but check *what they actually encode* before trusting them to color a map. Four checks, in order:

1. **PCA projection** — squash 1024 dims to 3 and render as RGB. If this produces a spatially coherent "semantic map" (not noise), the embeddings are capturing *something* structured.
2. **Correlation with hand-computed spectral indices** — NDVI (vegetation), NDBI (built-up), NDWI (water) are simple, fully interpretable band-math formulas. If PCA components correlate strongly with them, we know *what* structure the embeddings picked up.
3. **Agreement with ESA WorldCover** — an independent, human-labeled land-cover product. If unsupervised clusters over the embeddings line up with WorldCover classes, that's real external validation, not just an internally-consistent-looking picture.
4. **Nearest-neighbor sanity checks** — for a few chips we can eyeball (urban core, farmland, forest, water, the Marshall Fire burn scar), do the closest embeddings in the dataset actually look like the same kind of place?

Only after that does this notebook export the styled `docs/data/chips.geojson` + `docs/data/embeddings.bin` that the live map reads.

In [ ]:
REPO_URL = "https://github.com/<your-username>/front-range-embeddings.git"  # TODO: fill in

import os

if not os.path.exists("front-range-embeddings"):
    !git clone {REPO_URL} front-range-embeddings
%cd front-range-embeddings
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import sys

sys.path.append(os.getcwd())

import json

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
from scipy import stats
from sklearn.metrics import adjusted_rand_score

from src import stac_utils, viz_utils

embeddings = np.load("data/embeddings.npy")
chip_pixels = np.load("data/chip_pixels.npy")
with open("data/chips_meta.json") as f:
    chips_meta = json.load(f)

print(f"{embeddings.shape[0]} chips, {embeddings.shape[1]}-dim embeddings")
os.makedirs("outputs/figures", exist_ok=True)

## 1. PCA projection -> "semantic map" colors

In [ ]:
pca_colors, explained_variance = viz_utils.pca_to_hex_colors(embeddings)
print(f"Top-3 PCA components explain {explained_variance.sum()*100:.1f}% of variance")
print(f"Per-component: {[f'{v*100:.1f}%' for v in explained_variance]}")

In [ ]:
# Render the PCA colors back onto the (row, col) grid as a static preview --
# same information the interactive map shows, as a quick sanity image.
rows = [int(m["id"][1:4]) for m in chips_meta]
cols = [int(m["id"][5:8]) for m in chips_meta]
n_rows, n_cols = max(rows) + 1, max(cols) + 1

preview = np.zeros((n_rows, n_cols, 3), dtype=np.uint8)
for r, c, hexcolor in zip(rows, cols, pca_colors):
    preview[r, c] = [int(hexcolor[i : i + 2], 16) for i in (1, 3, 5)]

plt.figure(figsize=(10, 10))
plt.imshow(preview)
plt.title("Clay embeddings, PCA-projected to RGB")
plt.axis("off")
plt.savefig("outputs/figures/pca_semantic_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Correlate PCA components with hand-computed spectral indices

Band order in `chip_pixels` matches `stac_utils.S2_BANDS`: `[B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12]` (blue, green, red, ..., nir, ..., swir1, swir2).

In [ ]:
BLUE, GREEN, RED, NIR, SWIR1 = 0, 1, 2, 6, 8

def mean_index(pixels, band_a, band_b):
    a = pixels[:, band_a].mean(axis=(1, 2))
    b = pixels[:, band_b].mean(axis=(1, 2))
    return (a - b) / (a + b + 1e-6)

ndvi = mean_index(chip_pixels, NIR, RED)     # vegetation
ndbi = mean_index(chip_pixels, SWIR1, NIR)   # built-up
ndwi = mean_index(chip_pixels, GREEN, NIR)   # water

from sklearn.decomposition import PCA

pca3 = PCA(n_components=3, random_state=0).fit_transform(embeddings)

print(f"{'index':8s}  {'PC1':>8s}  {'PC2':>8s}  {'PC3':>8s}")
for name, index in [("NDVI", ndvi), ("NDBI", ndbi), ("NDWI", ndwi)]:
    corrs = [stats.pearsonr(pca3[:, i], index)[0] for i in range(3)]
    print(f"{name:8s}  {corrs[0]:8.2f}  {corrs[1]:8.2f}  {corrs[2]:8.2f}")

A strong correlation between a PCA component and NDVI/NDBI/NDWI means that axis of the embedding space is, in human terms, largely tracking vegetation/built-up-ness/water. Weak correlations across the board for a component that still explains real variance would suggest Clay is encoding something spectral indices don't capture (texture, context, the metadata conditioning) — worth calling out either way in the README.

## 3. Cluster the embeddings, check agreement with ESA WorldCover

In [ ]:
clusters = viz_utils.cluster_embeddings(embeddings, method="kmeans", n_clusters=8)
print(f"Cluster sizes: {np.bincount(clusters[clusters >= 0])}")

In [ ]:
WORLDCOVER_CLASSES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse", 70: "Snow/ice", 80: "Water",
    90: "Wetland", 95: "Mangroves", 100: "Moss/lichen",
}

catalog = stac_utils.open_catalog()
wc_items = stac_utils.search_worldcover(catalog)
wc_ds = odc.stac.load(
    wc_items, bbox=stac_utils.FRONT_RANGE_BBOX, crs="EPSG:32613",
    resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024},
)
wc_map = wc_ds.map.isel(time=0).compute().values if "time" in wc_ds.dims else wc_ds.map.compute().values

wc_grid = {c["id"]: c for c in stac_utils.make_pixel_chip_grid(*wc_map.shape)}

majority_class = []
matched_meta, matched_clusters = [], []
for meta, cluster in zip(chips_meta, clusters):
    win = wc_grid.get(meta["id"])
    if win is None:
        continue
    patch = wc_map[win["y_slice"], win["x_slice"]]
    values, counts = np.unique(patch[patch > 0], return_counts=True)
    if len(values) == 0:
        continue
    majority_class.append(values[np.argmax(counts)])
    matched_meta.append(meta)
    matched_clusters.append(cluster)

print(f"Matched {len(majority_class)}/{len(chips_meta)} chips to a WorldCover majority class")
ari = adjusted_rand_score(majority_class, matched_clusters)
print(f"Adjusted Rand Index (embedding clusters vs. WorldCover classes): {ari:.3f}")
print("(0 = no better than random agreement, 1 = perfect agreement -- unsupervised")
print(" clusters won't map 1:1 onto WorldCover's classes, so judge this relatively,")
print(" e.g. against a shuffled-label baseline, not against 1.0.)")

In [ ]:
import pandas as pd

class_names = [WORLDCOVER_CLASSES.get(c, str(c)) for c in majority_class]
contingency = pd.crosstab(pd.Series(matched_clusters, name="embedding cluster"),
                           pd.Series(class_names, name="WorldCover class"))

plt.figure(figsize=(10, 6))
plt.imshow(contingency.values, aspect="auto", cmap="viridis")
plt.xticks(range(len(contingency.columns)), contingency.columns, rotation=45, ha="right")
plt.yticks(range(len(contingency.index)), contingency.index)
plt.xlabel("ESA WorldCover class")
plt.ylabel("Embedding cluster")
plt.title("Embedding clusters vs. ground-truth land cover")
plt.colorbar(label="# chips")
plt.tight_layout()
plt.savefig("outputs/figures/cluster_vs_worldcover.png", dpi=150)
plt.show()
contingency

## 4. UMAP scatter — is the embedding space itself well-organized?

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=0)
umap_2d = reducer.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sc0 = axes[0].scatter(umap_2d[:, 0], umap_2d[:, 1], c=clusters, cmap="tab10", s=8)
axes[0].set_title("UMAP of Clay embeddings, colored by KMeans cluster")
plt.colorbar(sc0, ax=axes[0])

sc1 = axes[1].scatter(umap_2d[:, 0], umap_2d[:, 1], c=ndvi, cmap="RdYlGn", s=8)
axes[1].set_title("UMAP of Clay embeddings, colored by NDVI")
plt.colorbar(sc1, ax=axes[1])

plt.tight_layout()
plt.savefig("outputs/figures/umap_scatter.png", dpi=150)
plt.show()

## 5. Nearest-neighbor sanity checks

Pick a few chips we can independently characterize by their extreme spectral-index values (a proxy for "we're pretty sure what land-cover type this is"), plus the Marshall Fire burn scar by coordinates, and look at each one's nearest neighbors in embedding space alongside true-color thumbnails.

In [ ]:
def rgb_thumbnail(pixels):
    rgb = pixels[[RED, GREEN, BLUE]].astype("float32")
    return np.clip(rgb / (np.percentile(rgb, 98) + 1e-6), 0, 1).transpose(1, 2, 0)

def closest_chip_to(lat, lon):
    dists = [(m["lat"] - lat) ** 2 + (m["lon"] - lon) ** 2 for m in chips_meta]
    return int(np.argmin(dists))

cases = {
    "highest NDVI (likely forest)": int(np.argmax(ndvi)),
    "highest NDBI (likely urban)": int(np.argmax(ndbi)),
    "highest NDWI (likely water)": int(np.argmax(ndwi)),
    "lowest NDVI (likely bare/burned)": int(np.argmin(ndvi)),
    "Marshall Fire burn scar (Superior/Louisville, CO)": closest_chip_to(39.945, -105.14),
}

for label, query_idx in cases.items():
    neighbor_idxs = viz_utils.nearest_neighbors(embeddings, query_idx, k=5)
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    axes[0].imshow(rgb_thumbnail(chip_pixels[query_idx]))
    axes[0].set_title(f"query:\n{label}", fontsize=9)
    for ax, n_idx in zip(axes[1:], neighbor_idxs):
        ax.imshow(rgb_thumbnail(chip_pixels[n_idx]))
        ax.set_title(f"neighbor\n{chips_meta[n_idx]['id']}", fontsize=9)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    safe_name = label.split("(")[0].strip().replace(" ", "_").lower()
    plt.savefig(f"outputs/figures/neighbors_{safe_name}.png", dpi=150)
    plt.show()

## Export for the live map

In [ ]:
viz_utils.export_chips_geojson(chips_meta, pca_colors, clusters, ndvi, "docs/data/chips.geojson")
viz_utils.export_embeddings_bin(embeddings, "docs/data/embeddings.bin")

print("\nDone. Commit docs/data/chips.geojson, docs/data/embeddings.bin, and")
print("outputs/figures/*.png back to the repo, then open docs/index.html")
print("(or the GitHub Pages URL once enabled) to see the real map.")